# COD-20: 2025 B-IMD deprivation-cluster review

This notebook reproduces the clustering decision without rewriting tracked manifests and visualizes candidate quality, area assignments, and cluster profiles.

In [ ]:
import os
from pathlib import Path

current = Path.cwd().resolve()
project_root = next(
    path for path in (current, *current.parents)
    if (path / 'pyproject.toml').is_file()
)
os.chdir(project_root)
project_root

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from busan_imd.analysis.cluster_analysis import DEFAULT_PRIORITY_OUTPUT, FEATURE_COLUMNS, build

priority = pd.read_csv(DEFAULT_PRIORITY_OUTPUT, dtype={'admin_dong_code': str})
assignments, metrics, report = build(priority)
report['decision'], report['selected_cluster_count']

In [ ]:
quality = px.scatter(
    metrics,
    x='silhouette_score',
    y='mean_seed_stability_ari',
    size='minimum_cluster_size',
    color='passes_quality_gate',
    text='cluster_count',
    hover_data=['minimum_cluster_size', 'maximum_cluster_size', 'minimum_seed_stability_ari'],
    title='Cluster-candidate quality gate',
)
quality.add_vline(x=0.25, line_dash='dash', line_color='gray')
quality.add_hline(y=0.80, line_dash='dash', line_color='gray')
quality.update_traces(textposition='top center')
quality.show()

In [ ]:
scaled = StandardScaler().fit_transform(priority[list(FEATURE_COLUMNS)])
coordinates = PCA(n_components=2).fit_transform(scaled)
plot_data = priority[['admin_dong_code', 'sigungu_name', 'admin_dong_name']].copy()
plot_data['PC1'] = coordinates[:, 0]
plot_data['PC2'] = coordinates[:, 1]
plot_data = plot_data.merge(
    assignments[['admin_dong_code', 'b_imd_rank', 'cluster_id', 'cluster_label']],
    on='admin_dong_code',
    validate='one_to_one',
)
scatter = px.scatter(
    plot_data, x='PC1', y='PC2', color='cluster_label', symbol='cluster_id',
    text='admin_dong_name', hover_data=['sigungu_name', 'b_imd_rank'],
    title='2025 B-IMD priority-area deprivation types',
)
scatter.update_traces(textposition='top center', marker={'size': 11})
scatter.show()

In [ ]:
domain_labels = {
    'income': 'Income', 'employment': 'Employment', 'education': 'Education',
    'health': 'Health', 'housing_access': 'Housing and access',
    'living_environment': 'Living environment',
}
profile_data = {
    item['cluster_label']: item['mean_standardized_excess']
    for item in report['cluster_summaries']
}
profiles = pd.DataFrame(profile_data).T.rename(columns=domain_labels)
profile_heatmap = px.imshow(
    profiles, color_continuous_scale='RdBu_r', color_continuous_midpoint=0,
    text_auto='.2f', aspect='auto',
    title='Standardized domain-excess profile by deprivation type',
)
profile_heatmap.show()

In [ ]:
assignments[[
    'b_imd_rank', 'sigungu_name', 'admin_dong_name', 'cluster_id',
    'cluster_label', 'dominant_domain', 'secondary_domain',
]].sort_values(['cluster_id', 'b_imd_rank'])